# 🔬 Step 8: Exploratory Data Analysis (EDA) & Clinical Feature Analysis
### GluCare AI - Dataset Target Usia 20–40 Tahun (N = 653 Responden)

Notebook ini melakukan eksplorasi data komprehensif pada dataset kandidat training yang telah difrozen:
- **Populasi**: Dewasa Muda Usia 20–40 Tahun (Opsi A: Eksklusi Diabetes)
- **Komposisi Kelas**: 408 Bukan Prediabetes (`label = 0`) vs 245 Prediabetes (`label = 1`)
- **Tujuan**: Menguji apakah fitur-fitur klinis (glukosa puasa, BMI, lingkar perut, profil lipid, tekanan darah) memiliki pola pemisahan yang signifikan secara statistik antara kedua kelompok sebelum melangkah ke pemodelan Machine Learning.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Konfigurasi visualisasi
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

# Path dataset
DATA_PATH = Path('../data/processed/nhanes_glucare_target_20_40.csv')
df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded successfully! Shape: {df.shape}')
display(df.head(3))

## 1. Distribusi Kelas Target & Karakteristik Demografi
Memeriksa keseimbangan kelas antara **Bukan Prediabetes (0)** dan **Prediabetes (1)** serta distribusi usia dan jenis kelamin.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Target Class Distribution
ax1 = axes[0]
sns.countplot(data=df, x='ground_truth_status', palette=['#3182CE', '#DD6B20'], ax=ax1)
ax1.set_title('Distribusi Kelas Target (Step 6)')
ax1.set_xlabel('Status Klinis')
ax1.set_ylabel('Jumlah Sampel')
for p in ax1.patches:
    h = p.get_height()
    ax1.annotate(f'{int(h)} ({h/len(df)*100:.1f}%)', (p.get_x() + p.get_width()/2., h),
                 ha='center', va='bottom', xytext=(0, 4), textcoords='offset points', fontweight='bold')

# 2. Age Distribution by Target
ax2 = axes[1]
sns.histplot(data=df, x='age', hue='ground_truth_status', kde=True, bins=15, palette=['#3182CE', '#DD6B20'], ax=ax2)
ax2.set_title('Distribusi Usia (20–40 Tahun)')
ax2.set_xlabel('Usia (Tahun)')
ax2.set_ylabel('Frekuensi')

# 3. Gender Distribution by Target
ax3 = axes[2]
gender_df = df.copy()
gender_df['gender_label'] = gender_df['gender'].map({1: 'Laki-laki', 2: 'Perempuan'})
sns.countplot(data=gender_df, x='gender_label', hue='ground_truth_status', palette=['#3182CE', '#DD6B20'], ax=ax3)
ax3.set_title('Distribusi Jenis Kelamin')
ax3.set_xlabel('Jenis Kelamin')
ax3.set_ylabel('Jumlah')

plt.tight_layout()
plt.show()

## 2. Uji Statistik Komparatif: Bukan Prediabetes vs Prediabetes
Menghitung rata-rata, standar deviasi, median, dan uji signifikansi **Mann-Whitney U** serta ukuran efek (**Cohen's d**) untuk setiap fitur klinis.

In [ ]:
from src.eda_analysis import perform_statistical_comparison

comp_df = perform_statistical_comparison(df)
print('Tabel Perbandingan Statistik Fitur Klinis (Urut dari Effect Size Terbesar):')
display(comp_df[['feature', 'mean_bukan_prediabetes', 'mean_prediabetes', 'difference_mean', 'cohens_d', 'p_value', 'significant_005']])

## 3. Visualisasi Distribusi Fitur Kunci Metabolik & Antropometri
Membandingkan boxplot dan kurva kepadatan (*KDE*) untuk fitur yang paling diskriminatif:

In [ ]:
key_features = ['glucose_fasting', 'waist_cm', 'bmi', 'tg_hdl_ratio', 'triglycerides', 'hdl', 'bp_systolic', 'map_pressure']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    sns.boxplot(data=df, x='ground_truth_status', y=feat, palette=['#63B3ED', '#F6AD55'], ax=axes[i], width=0.5)
    axes[i].set_title(f'Pola {feat}', fontweight='bold')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

## 4. Matriks Korelasi & Analisis Multikolinearitas
Menganalisis hubungan antar variabel prediktor dan korelasinya terhadap label prediabetes.

In [ ]:
clinical_features = [
    'glucose_fasting', 'bmi', 'waist_cm', 'weight', 'height',
    'hdl', 'triglycerides', 'tg_hdl_ratio',
    'bp_systolic', 'bp_diastolic', 'map_pressure',
    'age', 'label_prediabetes'
]

corr_matrix = df[clinical_features].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='vlag', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'})
plt.title('Matriks Korelasi Fitur Klinis GluCare (Target Usia 20–40)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Kesimpulan Klinis & Rekomendasi Fitur untuk Modeling (Step 9–10)
Berdasarkan analisis di atas:
1. **Glukosa Puasa (`glucose_fasting`)**: Memiliki korelasi dan separasi terkuat (rata-rata 103.5 vs 88.5 mg/dL).
2. **Adipositas Sentral (`waist_cm` & `bmi`)**: Responden prediabetes memiliki lingkar perut dan BMI yang secara signifikan lebih tinggi ($p < 0.001$).
3. **Profil Lipid & Rasio Aterogenik (`tg_hdl_ratio`, `triglycerides`, `hdl`)**: Rasio TG/HDL dan trigliserida lebih tinggi, sementara HDL lebih rendah pada prediabetes ($p < 0.001$).
4. **Tekanan Darah (`map_pressure`, `bp_systolic`, `bp_diastolic`)**: Terdapat korelasi positif yang signifikan dengan risiko prediabetes ($p < 0.01$).
5. **Multikolinearitas**: `weight` dan `bmi` berkorelasi sangat kuat ($r > 0.85$), sehingga `bmi` dan `waist_cm` lebih direkomendasikan sebagai representasi proporsi tubuh tanpa redundansi.